In [1]:
import polars as pl
import pathlib
import sys
import json
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

In [2]:
sys.path.append('/Users/urolyi1/repos/march-madness')

In [3]:
import kaggle_2026.preprocess_data as preprocess_data
import uro_cbb.model_utils as model_utils
from uro_cbb.preprocess import merge_with_kaggle_names

In [4]:
%load_ext autoreload
%reload_ext autoreload
%autoreload 2

## Preprocess historical data for training

In [5]:
preprocessed_dfs = {}
for year in range(2015, 2026):
    if year == 2020:
        continue
    preprocessed_dfs[year] = preprocess_data.preprocess_data(year)

ERROR:root:Missing names: {'arkansas pine bluff', 'cal st. bakersfield', 'ut rio grande valley', 'tennessee martin', 'illinois chicago', 'louisiana monroe', 'mississippi valley st.', 'st. francis ny', 'bethune cookman', 'texas a&m corpus chris', 'saint francis', 'southeast missouri st.'}
ERROR:root:Missing names: {'arkansas pine bluff', 'cal st. bakersfield', 'tennessee martin', 'illinois chicago', 'louisiana monroe', 'mississippi valley st.', 'st. francis ny', 'st. francis pa', 'bethune cookman', 'arkansas little rock', 'texas a&m corpus chris', 'texas pan american', 'louisiana lafayette', 'southeast missouri st.'}
ERROR:root:Missing names: {'arkansas pine bluff', 'cal st. bakersfield', 'ut rio grande valley', 'tennessee martin', 'illinois chicago', 'louisiana monroe', 'mississippi valley st.', 'st. francis ny', 'bethune cookman', 'texas a&m corpus chris', 'saint francis', 'southeast missouri st.'}
ERROR:root:Missing names: {'arkansas pine bluff', 'cal st. bakersfield', 'ut rio grande

In [6]:
womens_preprocessed_dfs = {}
for year in range(2016, 2026):
    if year == 2020:
        continue
    womens_preprocessed_dfs[year] = preprocess_data.preprocess_womens_data(year)

ERROR:root:Missing names: {'arkansas pine bluff', 'cal st. bakersfield', 'ut rio grande valley', 'tennessee martin', 'illinois chicago', 'louisiana monroe', 'mississippi valley st.', 'bethune cookman', 'texas a&m corpus chris', 'texas a&m commerce', 'saint francis', 'queens', 'southeast missouri st.', 'tarleton st.'}
ERROR:root:Missing names: {'arkansas pine bluff', 'cal st. bakersfield', 'ut rio grande valley', 'tennessee martin', 'illinois chicago', 'louisiana monroe', 'mississippi valley st.', 'bethune cookman', 'texas a&m corpus chris', 'texas a&m commerce', 'saint francis', 'queens', 'southeast missouri st.', 'tarleton st.'}
ERROR:root:Missing names: {'arkansas pine bluff', 'cal st. bakersfield', 'ut rio grande valley', 'tennessee martin', 'illinois chicago', 'louisiana monroe', 'mississippi valley st.', 'bethune cookman', 'texas a&m corpus chris', 'texas a&m commerce', 'saint francis', 'queens', 'southeast missouri st.', 'tarleton st.'}
ERROR:root:Missing names: {'arkansas pine b

## Train and save models

In [8]:
training_df = pl.concat([womens_preprocessed_dfs[year] for year in range(2016, 2026) if year != 2020], how='diagonal')
training_df = training_df.to_pandas()
x_cols = [column for column in training_df.columns[8:32] if '_2' not in column]
x_2_cols = [column for column in training_df.columns if '_2' in column]
assert len(x_cols) == len(x_2_cols)
dup_training_df = training_df.copy()
dup_training_df[x_cols] = training_df[x_2_cols]
dup_training_df[x_2_cols] = training_df[x_cols]
dup_training_df['Result'] = 1 - training_df['Result']
final_training_df = pd.concat([training_df, dup_training_df]).reset_index(drop=True)
womens_model = smf.glm(
    "Result ~ adjoe + adjoe_2 + adjde + adjde_2 ",
    data=final_training_df,
    family=sm.families.Binomial(link=sm.families.links.Logit())
)
womens_results = womens_model.fit()
womens_results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                 Result   No. Observations:                 1124
Model:                            GLM   Df Residuals:                     1119
Model Family:                Binomial   Df Model:                            4
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -552.61
Date:                Thu, 19 Mar 2026   Deviance:                       1105.2
Time:                        11:16:58   Pearson chi2:                 1.17e+03
No. Iterations:                     5   Pseudo R-squ. (CS):             0.3317
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -1.066e-14      2.601   -4.1e-15      1.000      -5.098       5.098
adjoe          0.0756      0.010      7.202      0.000       0.055       0.096
adjoe_2       -0.0756      0.010     -7.202      0.000      -0.096      -0.055
adjde         -0.0820      0.014     -5.864      0.000      -0.109      -0.055
adjde_2        0.0820      0.014      5.864      0.000       0.055       0.109
==============================================================================
"""

In [9]:
womens_results.save('../kaggle_2026/models/womens_barttorvik.pkl')

In [18]:
training_df = pl.concat([preprocessed_dfs[year] for year in range(2016, 2026) if year != 2020], how='diagonal')
training_df = training_df.to_pandas()
x_cols = [column for column in training_df.columns[8:] if '_2' not in column]
x_2_cols = [column for column in training_df.columns if '_2' in column]
assert len(x_cols) == len(x_2_cols)
dup_training_df = training_df.copy()
dup_training_df[x_cols] = training_df[x_2_cols]
dup_training_df[x_2_cols] = training_df[x_cols]
dup_training_df['Result'] = 1 - training_df['Result']
final_training_df = pd.concat([training_df, dup_training_df]).reset_index(drop=True)
mens_model = smf.glm(
    "Result ~ adjoe + adjoe_2 + adjde + adjde_2 + AdjO + AdjO_2 + AdjD + AdjD_2",
    data=final_training_df,
    family=sm.families.Binomial(link=sm.families.links.Logit())
)
mens_results = mens_model.fit()
mens_results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                 Result   No. Observations:                 1128
Model:                            GLM   Df Residuals:                     1123
Model Family:                Binomial   Df Model:                            4
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -611.51
Date:                Thu, 19 Mar 2026   Deviance:                       1223.0
Time:                        11:18:03   Pearson chi2:                 1.18e+03
No. Iterations:                     5   Pseudo R-squ. (CS):             0.2607
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   1.599e-14      2.796   5.72e-15      1.000      -5.481       5.481
adjoe          0.1056      0.013      8.426      0.000       0.081       0.130
adjoe_2       -0.1056      0.013     -8.426      0.000      -0.130      -0.081
adjde         -0.1007      0.017     -5.998      0.000      -0.134      -0.068
adjde_2        0.1007      0.017      5.998      0.000       0.068       0.134
==============================================================================
"""

In [19]:
mens_results.save('../kaggle_2026/models/mens_barttorvik_kenpom.pkl')

## Clean and save 2026 team data

In [20]:
module_dir = pathlib.Path('.').absolute()
kaggle_names_df = pl.read_csv(
    module_dir.parent / "data/kaggle_2026/raw/MTeamSpellings.csv",
    schema={"Team": pl.Utf8, "TeamID": pl.Int16},
).join(
    pl.read_csv(module_dir.parent / "data/kaggle_2026/raw/MTeams.csv").select(
        pl.col("TeamID"), pl.col("TeamName")
    ),
    on="TeamID",
)
kaggle_names_df

Team,TeamID,TeamName
str,i16,str
"""a&m-corpus chris""",1394,"""TAM C. Christi"""
"""a&m-corpus christi""",1394,"""TAM C. Christi"""
"""abilene chr""",1101,"""Abilene Chr"""
"""abilene christian""",1101,"""Abilene Chr"""
"""abilene-christian""",1101,"""Abilene Chr"""
…,…,…
"""youngstown st""",1464,"""Youngstown St"""
"""youngstown st.""",1464,"""Youngstown St"""
"""youngstown state""",1464,"""Youngstown St"""


In [21]:
kenpom_df =(
    pl.read_parquet('../data/kenpom/raw/kenpom_2026.parquet')
    .with_columns(
        pl.col("Team").str.to_lowercase().alias("Team")
    )
    .pipe(merge_with_kaggle_names, kaggle_names_df, "Team")
    .select(
        pl.col('Team'),
        pl.col('Team_id').alias('TeamID'),
        pl.col('AdjO'),
        pl.col('AdjD')
    )
)
kenpom_df

ERROR:root:Missing names: {'arkansas pine bluff', 'cal st. bakersfield', 'ut rio grande valley', 'louisiana monroe', 'illinois chicago', 'tennessee martin', 'mississippi valley st.', 'bethune cookman', 'southeast missouri', 'texas a&m corpus chris', 'saint francis', 'queens', 'tarleton st.'}


Team,TeamID,AdjO,AdjD
str,i16,f64,f64
"""Abilene Chr""",1101,101.1,107.6
"""Air Force""",1102,95.6,115.5
"""Akron""",1103,118.8,106.0
"""Alabama""",1104,129.0,103.3
"""Alabama A&M""",1105,104.0,116.7
…,…,…,…
"""Wright St""",1460,112.1,110.0
"""Wyoming""",1461,115.1,107.7
"""Xavier""",1462,116.9,109.4


In [22]:
barttorvik_df = (
    pl.read_parquet('../data/barttorvik/raw/mens/barttorvik_2026.parquet')
    .with_columns(
        pl.col("team").str.to_lowercase().alias("team")
    )
    .pipe(merge_with_kaggle_names, kaggle_names_df, "team")
    .select(
        pl.col('team').alias('Team'),
        pl.col('team_id').alias('TeamID'),
        pl.col('adjoe'),
        pl.col('adjde')
    )
)
barttorvik_df

ERROR:root:Missing names: {'arkansas pine bluff', 'cal st. bakersfield', 'ut rio grande valley', 'tennessee martin', 'illinois chicago', 'louisiana monroe', 'mississippi valley st.', 'bethune cookman', 'texas a&m corpus chris', 'saint francis', 'queens', 'southeast missouri st.', 'tarleton st.'}


Team,TeamID,adjoe,adjde
str,i16,f32,f32
"""Abilene Chr""",1101,99.900002,107.0
"""Air Force""",1102,95.699997,112.900002
"""Akron""",1103,117.900002,105.900002
"""Alabama""",1104,129.600006,102.300003
"""Alabama A&M""",1105,103.400002,115.800003
…,…,…,…
"""Wright St""",1460,111.900002,108.099998
"""Wyoming""",1461,114.099998,106.199997
"""Xavier""",1462,115.800003,107.900002


In [23]:
cleaned_df = kenpom_df.join(
    barttorvik_df,
    on=['Team', 'TeamID'],
    how='inner'
)
cleaned_df

Team,TeamID,AdjO,AdjD,adjoe,adjde
str,i16,f64,f64,f32,f32
"""Abilene Chr""",1101,101.1,107.6,99.900002,107.0
"""Air Force""",1102,95.6,115.5,95.699997,112.900002
"""Akron""",1103,118.8,106.0,117.900002,105.900002
"""Alabama""",1104,129.0,103.3,129.600006,102.300003
"""Alabama A&M""",1105,104.0,116.7,103.400002,115.800003
…,…,…,…,…,…
"""Wright St""",1460,112.1,110.0,111.900002,108.099998
"""Wyoming""",1461,115.1,107.7,114.099998,106.199997
"""Xavier""",1462,116.9,109.4,115.800003,107.900002


In [24]:
out_dir = pathlib.Path('../data/kaggle_2026/cleaned/mens/')
out_dir.mkdir(parents=True, exist_ok=True)
cleaned_df.write_csv(out_dir / 'barttorvik_kenpom.csv')
cleaned_df.write_parquet(out_dir / 'barttorvik_kenpom.parquet')

## Clean and save 2026 women's team data

In [25]:
kaggle_womens_names_df = pl.read_csv(
    module_dir.parent / "data/kaggle_2026/raw/WTeamSpellings.csv",
    schema={"Team": pl.Utf8, "TeamID": pl.Int16},
).join(
    pl.read_csv(module_dir.parent / "data/kaggle_2026/raw/WTeams.csv").select(
        pl.col("TeamID"), pl.col("TeamName")
    ),
    on="TeamID",
)
kaggle_womens_names_df

Team,TeamID,TeamName
str,i16,str
"""a&m-corpus chris""",3394,"""TAM C. Christi"""
"""a&m-corpus christi""",3394,"""TAM C. Christi"""
"""abilene chr""",3101,"""Abilene Chr"""
"""abilene christian""",3101,"""Abilene Chr"""
"""abilene-christian""",3101,"""Abilene Chr"""
…,…,…
"""youngstown st""",3464,"""Youngstown St"""
"""youngstown st.""",3464,"""Youngstown St"""
"""youngstown state""",3464,"""Youngstown St"""


In [26]:
barttorvik_df = (
    pl.read_parquet('../data/barttorvik/raw/womens/barttorvik_2026.parquet')
    .with_columns(
        pl.col("team").str.to_lowercase().alias("team")
    )
    .pipe(merge_with_kaggle_names, kaggle_womens_names_df, "team")
    .select(
        pl.col('team').alias('Team'),
        pl.col('team_id').alias('TeamID'),
        pl.col('adjoe'),
        pl.col('adjde')
    )
)
barttorvik_df

ERROR:root:Missing names: {'arkansas pine bluff', 'cal st. bakersfield', 'ut rio grande valley', 'tennessee martin', 'illinois chicago', 'louisiana monroe', 'mississippi valley st.', 'bethune cookman', 'texas a&m corpus chris', 'saint francis', 'queens', 'southeast missouri st.', 'tarleton st.'}


Team,TeamID,adjoe,adjde
str,i16,f32,f32
"""Abilene Chr""",3101,97.0,89.900002
"""Air Force""",3102,89.199997,93.300003
"""Akron""",3103,90.099998,106.400002
"""Alabama""",3104,109.699997,80.599998
"""Alabama A&M""",3105,82.5,89.599998
…,…,…,…
"""Wright St""",3460,88.400002,102.900002
"""Wyoming""",3461,85.400002,92.099998
"""Xavier""",3462,91.699997,88.800003


In [27]:
out_dir = pathlib.Path('../data/kaggle_2026/cleaned/womens/')
out_dir.mkdir(parents=True, exist_ok=True)
barttorvik_df.write_csv(out_dir / 'barttorvik.csv')
barttorvik_df.write_parquet(out_dir / 'barttorvik.parquet')